In [ ]:
import pandas as pd

In [ ]:
rows_changed = {}

CATEGORY_MAP = {
    'Emergency Supplies': 'Emergency',
    'Groceries': 'Grocery',
    'Games & Entertainment': 'Entertainment',
    'Household Essentials': 'Household',
    'Baby Products': 'Baby',
    'Breakfast Foods': 'Breakfast',
    'Alcoholic Beverages': 'Alcohol',
    'Health & Wellness': 'Health',
}


def clean_store_id(col):
    col = col.str.upper().str.strip()
    col = col.str.replace('-', '').str.replace('_', '').str.replace(' ', '')
    col = 'FL-' + col.str[2:]
    return col


def clean_transactions(df):
    rows_changed['duplicates'] = df.duplicated().sum()
    df = df.drop_duplicates().copy()

    df['unit_price'] = df['unit_price'].str.replace('$', '')
    df['unit_price'] = df['unit_price'].astype(float)
    rows_changed['unit_price'] = len(df)

    rows_changed['quantity'] = (df['quantity'] < 0).sum()
    df = df[df['quantity'] >= 1].copy()

    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', errors='coerce')
    rows_changed['timestamp'] = df['timestamp'].isna().sum()
    df = df[df['timestamp'].notna()].copy()

    before = df['store_id'].copy()
    df['store_id'] = clean_store_id(before)
    rows_changed['store_id'] = int(before.ne(df['store_id']).sum())

    before = df['category'].copy()
    df['category'] = (before.str.strip().str.title().replace(CATEGORY_MAP))
    rows_changed['category'] = int(before.ne(df['category']).sum())

    return df


def clean_stores(df):
    df = df.copy()
    df['store_id'] = clean_store_id(df['store_id'])

    df['state'] = (
        df['state'].str.strip().str.upper()
        .replace({'FLORIDA': 'FL', 'FLA.': 'FL'})
    )
    df['region'] = df['region'].str.strip().str.title()

    rows_changed['stores'] = int(
        df.duplicated('store_id').sum()
    )
    df = df.drop_duplicates('store_id').copy()

    df['has_coordinates'] = (
        df[['latitude', 'longitude']].notna().all(axis=1)
    )
    return df


transactions_raw = pd.read_csv('../../data/walmart_transactions.csv')
stores_raw = pd.read_csv('../../data/store_locations.csv')

transactions_typed = clean_transactions(transactions_raw)
stores = clean_stores(stores_raw)

print('rows before:', len(transactions_raw))
print('rows after:', len(transactions_typed))
print('rows removed:', len(transactions_raw) - len(transactions_typed))
print('store_ids:', transactions_typed['store_id'].nunique())
print('categories:', transactions_typed['category'].nunique())
print('stores in sales but not in the store file:', set(transactions_typed['store_id']) - set(stores['store_id']))

In [ ]:
DECISIONS = [
    {'step': 'duplicates', 'decision': 'dropped exact duplicate rows', 'rows': rows_changed['duplicates']},
    {'step': 'unit_price', 'decision': 'removed $ and turned price from text into a number', 'rows': rows_changed['unit_price']},
    {'step': 'quantity', 'decision': 'dropped rows with negative quantity as data entry errors', 'rows': rows_changed['quantity']},
    {'step': 'timestamp', 'decision': 'parsed the different date formats and dropped rows with no timestamp', 'rows': rows_changed['timestamp']},
    {'step': 'store_id', 'decision': 'changed every store_id to the format FL-123 so it joins to the store file', 'rows': rows_changed['store_id']},
    {'step': 'category', 'decision': 'combined the different category spellings into one label each', 'rows': rows_changed['category']},
    {'step': 'stores', 'decision': 'cleaned store_id in the store file and dropped stores listed twice', 'rows': rows_changed['stores']},
]

decision_log = pd.DataFrame(DECISIONS)
print('decisions logged:', len(decision_log))
decision_log

In [ ]:
#Work in progress

CANON = {
    # Pop-Tarts
    'POPTART-STRAW':  'POPTART-STRAWBERRY',
    'PT-12':          'POPTART-STRAWBERRY',
    'SKU#459812':     'POPTART-STRAWBERRY',
    'PT_STRAWBERRY':  'POPTART-STRAWBERRY',
    # bottled water
    'WTR-24':         'WATER-24PK',
    'WATER24PK':      'WATER-24PK',
    'WTR_24PK':       'WATER-24PK',
    # Monopoly
    'BG-MONO':        'MONOPOLY',
    'MONOPOLY-01':    'MONOPOLY',
    'GAME-MONO':      'MONOPOLY',
    # AA batteries
    'BAT-AA4':        'AA-BATTERIES-4PK',
    'BATT_AA_4PK':    'AA-BATTERIES-4PK',
    'BATT-100':       'AA-BATTERIES-4PK',
    # two SKUs each
    'BEER-24':        'LIGHT-BEER-24PK',
    'LTBEER24':       'LIGHT-BEER-24PK',
    'BREAD-WH':       'WHITE-BREAD',
    'WHITEBREAD':     'WHITE-BREAD',
    'PB-16':          'PEANUT-BUTTER',
    'PNUTBTR-16':     'PEANUT-BUTTER',
    'CNDL-EM':        'EMERGENCY-CANDLES',
    'CANDLE-EMRG':    'EMERGENCY-CANDLES',
    'SOUP-CAN':       'CANNED-SOUP',
    'CNSOUP-01':      'CANNED-SOUP',
    'CARDS-01':       'PLAYING-CARDS',
    'PLAYCARD-STD':   'PLAYING-CARDS',
    'CHIPS-VP':       'CHIPS-VARIETY',
    'CHIPVARIETY':    'CHIPS-VARIETY',
    'TARP-HD':        'HEAVY-DUTY-TARP',
    'HDTARP-01':      'HEAVY-DUTY-TARP',
    'FLASH-01':       'LED-FLASHLIGHT',
    'LED-FLASH':      'LED-FLASHLIGHT',
    'PTOWEL-6':       'PAPER-TOWELS',
    'PAPERTOWEL6':    'PAPER-TOWELS',
    'FAID-01':        'FIRST-AID-KIT',
    'FIRSTAID-KIT':   'FIRST-AID-KIT',
    'TP-12':          'TOILET-PAPER-12',
    'TOILETPAPER12':  'TOILET-PAPER-12',
    'DIAP-S3':        'DIAPERS-SIZE3',
    'DIAPERS-3':      'DIAPERS-SIZE3',
}

sales = transactions_typed.copy()
sales['canonical_sku'] = sales['sku'].map(CANON).fillna(sales['sku'])

print('rows:', len(sales))
print('nulls in canonical_sku:', sales['canonical_sku'].isna().sum())
print('distinct SKUs before:', sales['sku'].nunique(), 'after:', sales['canonical_sku'].nunique())
print()

group_sizes = sales.groupby('canonical_sku')['sku'].nunique().sort_values(ascending=False)
print('largest group:', group_sizes.index[0], 'with', group_sizes.iloc[0], 'SKUs')
print(sales[sales['canonical_sku'] == 'POPTART-STRAWBERRY'][['sku', 'product_name', 'canonical_sku']].drop_duplicates('sku').to_string(index=False))

In [ ]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    if tries < 1:
        raise ValueError('tries must be at least 1')
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue

        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                r.raise_for_status()
            time.sleep(2 ** attempt)
            continue

        r.raise_for_status()
        return pd.DataFrame(r.json()['daily'])


wx = get_weather('2024-09-01', '2024-09-30')

wx = wx.rename(columns={
    'time': 'date',
    'precipitation_sum': 'precip_mm',
    'windspeed_10m_max': 'wind_max_kmh',
    'pressure_msl_mean': 'pressure_hpa',
})
wx['date'] = pd.to_datetime(wx['date'])
wx['source'] = 'open-meteo'

print('rows:', len(wx))
wx.head()

In [ ]:
import matplotlib.pyplot as plt

daily = transactions_typed.copy()
daily['date'] = daily['timestamp'].dt.normalize()
daily = daily.groupby('date')['transaction_id'].count()

plt.figure(figsize=(12,5))
plt.plot(daily.index, daily.values)
plt.title('Daily order volume, September 2024')
plt.xlabel('Date')
plt.ylabel('Orders per day')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()